## Understanding How Claude Uses Tools

# Unit 22: Ingesting Tool Schemas and Parsing Tool Use Payloads

## Introduction & Overview

Welcome to understanding how Claude responds when it decides to utilize your tools! In the previous lesson, you learned how to create tool schemas that describe your functions to Claude. Now you will discover how to include these tools in your API requests and, more importantly, how to interpret Claude's responses when it wants to use them.

In this lesson, you'll learn how to configure your requests to enable tool use, understand the different types of responses Claude can provide, and extract the specific information you need to execute the tools Claude requests. By the end, you'll be able to recognize when Claude wants to use a tool and gather all the details needed to make that happen.

---

## Adding Tool Guidance to Your System Prompt

While Claude will automatically see and can use any tools you provide in your request, mentioning tools in your system prompt can help ensure more consistent behavior and guide Claude toward using them appropriately.

Here's an example of how you can improve your system prompt:

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# System prompt with optional tool usage guidance
system_prompt = (
    "You are a helpful math assistant. "
    "When performing calculations, use the available tools for accuracy."
)

```

This guidance helps Claude understand your preferences for when and how to use tools, but it's not required for tool functionality. Claude can still recognize and use tools based solely on their availability in the request.

---

## Providing Tools to Claude

The `tools` parameter is the essential component that makes your functions available to Claude. This parameter accepts the JSON array of schemas you created in the previous lesson:

```python
import json

# Load your tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a message requesting a calculation
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# Send the request with tools enabled
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas  # This makes your tools available to Claude
)

```

Claude will recognize the available tools from the `tools` parameter alone. The system prompt guidance can be helpful for encouraging consistent tool usage patterns, but Claude will still be able to see and use your tools even without explicit instructions in the system prompt.

---

## Understanding Claude's Tool Use Responses

When Claude decides to use a tool, the response structure changes significantly from a simple text response. Let's examine what a complete tool use response looks like by printing the entire response structure:

```python
# Print the complete response structure
print(json.dumps(response.model_dump(), indent=2))

```

This will output a detailed JSON structure showing all the components of Claude's response:

```json
{
  "id": "msg_01LioNGfNM9nEiE1peb9ixAu",
  "content": [
    {
      "text": "I'll calculate 15 + 27 for you using the sum function.",
      "type": "text"
    },
    {
      "id": "toolu_01ErJzJmxjYisApSybjcTcs4",
      "input": {
        "a": 15,
        "b": 27
      },
      "name": "sum_numbers",
      "type": "tool_use"
    }
  ],
  "model": "claude-sonnet-4-6",
  "role": "assistant",
  "stop_reason": "tool_use",
  "type": "message",
  "usage": {
    "input_tokens": 425,
    "output_tokens": 85
  }
}

```

Notice two key differences from regular text responses:

* **The `stop_reason` is `"tool_use"**` instead of `"end_turn"`.
* **The `content` array contains both text and tool use blocks.**

This structure allows Claude to explain what it's doing while providing the structured information your system needs to execute the requested functions.

---

## The Importance of `stop_reason`

The `stop_reason` field is your primary indicator for determining what action to take next in your agent logic. Let's extract and examine this field to understand its possible values:

```python
# Check the stop reason to determine next steps
print(f"Stop Reason: {response.stop_reason}")

```

When you run this code, you'll see the following output:

```text
Stop Reason: tool_use

```

The `stop_reason` acts as a control flow signal for your agent. Here are the main values you'll encounter:

| `stop_reason` | System Meaning & Control Flow |
| --- | --- |
| **`"tool_use"`** | Claude has requested tool execution and is waiting for results before continuing. |
| **`"end_turn"`** | Claude has completed its response and the conversation can continue normally. |
| **`"max_tokens"`** | Claude reached the token limit and was cut off before finishing its thought. |
| **`"stop_sequence"`** | Claude encountered a predefined custom stop sequence string. |

Understanding these values helps you build proper agent logic that responds appropriately to Claude's intentions.

---

## Understanding the Content Array and Tool Use Blocks

When `stop_reason` is `"tool_use"`, the `content` array contains a mix of text explanations and tool use requests. Claude includes text blocks to explain its reasoning and what it's about to do, making the interaction more transparent and user-friendly.

The `content` array can also contain multiple tool use blocks when Claude wants to execute several tools in parallel. Each content item has a `type` field that tells you how to handle it. Let's iterate through the content array to examine each item:

```python
# Process each content item
for i, content_item in enumerate(response.content):
    print(f"\nContent Item {i+1}:")
    print(f"Type: {content_item.type}")
        
    if content_item.type == "text":
        print(f"Text: {content_item.text}")
    elif content_item.type == "tool_use":
        print(f"Tool Name: {content_item.name}")
        print(f"Tool Input: {content_item.input}")
        print(f"Tool ID: {content_item.id}")

```

Running this code will show you the structure of each content item:

```text
Content Item 1:
Type: text
Text: I'll calculate 15 + 27 for you using the sum function.

Content Item 2:
Type: tool_use
Tool Name: sum_numbers
Tool Input: {'a': 15, 'b': 27}
Tool ID: toolu_01ErJzJmxjYisApSybjcTcs4

```

Each tool use block contains three essential pieces of information that your system needs to execute the requested function:

* **`name`:** The function name that matches your tool schema (e.g., `"sum_numbers"`).
* **`input`:** A dictionary of parameters to pass to your function, with keys matching your schema's parameter names.
* **`id`:** A unique identifier string for this specific tool use request, which you'll need when sending results back to Claude.

These fields work together to provide everything you need to execute the tool and maintain proper conversation flow.

---

## Summary and Next Steps

You now understand how Claude communicates its tool use intentions through structured API responses. When Claude decides to use tools, it provides both human-readable explanations and machine-readable tool use blocks within the `content` array, with the `stop_reason` field acting as a control signal for your system.

In the upcoming practice exercises, you'll work with these response structures hands-on, learning to parse tool use requests and prepare for the next step: actually executing the requested tools and sending results back to Claude to complete the conversation flow.

## Making Tools Available to Claude

Now that you understand how to create tool schemas, it's time to see what happens when you actually make those tools available to Claude! In this exercise, you'll transform a basic API request into one that enables tool usage, allowing you to observe how Claude's response structure changes when tools are present.

Currently, your code makes a simple request to Claude asking for a calculation, but Claude doesn't have any available tools. You'll need to:

    Load the tool schemas from the existing schemas.json file
    Add the tools parameter to your API request

This hands-on comparison will help you understand the immediate impact of providing tools to Claude and prepare you for the next steps in building tool-enabled agents!

```
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# System prompt describing Claude as a math assistant
system_prompt = (
    "You are a helpful math assistant."
)

# TODO: Load the tool schemas from the schemas.json file

# Create a message requesting a calculation
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# Send the request to Claude
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    messages=messages,
    system=system_prompt
    # TODO: Add the tools parameter with the loaded schemas
)

# Print the complete response structure
print(json.dumps(response.model_dump(), indent=2))


```

To transition your standard conversational integration into an active tool orchestration ring, you need to map your external JSON schema manifest straight into the Anthropic message generation engine.

Passing your array of schemas into the `tools` keyword parameter instantly exposes your backend functions to Claude's semantic routing layer.

Here is the complete script with your JSON loading operations and tool API parameters fully implemented:

```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# System prompt describing Claude as a math assistant
system_prompt = (
    "You are a helpful math assistant."
)

# Load the tool schemas from the schemas.json file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a message requesting a calculation
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# Send the request to Claude with tools explicitly enabled
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas  # Register the loaded schemas into Claude's context window
)

# Print the complete response structure to examine the payload mutation
print(json.dumps(response.model_dump(), indent=2))

```

### What to Look for in the Output:

Once you execute this code, look closely at the console printout. You'll observe a profound shift in the data contract returned by the server:

* **The `stop_reason` Shift:** Instead of ending with `"stop_reason": "end_turn"`, the transaction will explicitly return `"stop_reason": "tool_use"`.
* **The Ingestion Block:** Inside the `content` array, you will now see a structured dictionary object featuring `"type": "tool_use"`, passing back an explicitly generated unique `"id"`, the string target `"name": "sum_numbers"`, and beautifully structured numerical arguments: `{"a": 15, "b": 27}`.

This change flags your local runtime environment to stop talking and start executing your backend Python math logic! 🚀

## Guiding Claude with System Prompts

Great work on making tools available to Claude! Now it's time to discover how your system prompt can influence Claude's decision-making when those tools are present.

While Claude can automatically recognize and use any tools you provide, the way you write your system prompt can significantly impact how consistently and appropriately Claude chooses to use them. In this exercise, your mission is to enhance a basic system prompt to better guide Claude toward using available tools for mathematical calculations.

This small change will help you shape Claude's behavior and make your tool-enabled agents more predictable and effective!

```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# TODO: Enhance this system prompt to guide Claude toward using tools for calculations
system_prompt = (
    "You are a helpful assistant."
)

# Load the tool schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a message requesting a calculation
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# Send the request with tools enabled
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas
)

# Print the complete response structure
print(json.dumps(response.model_dump(), indent=2))

```

While Claude can recognize and invoke tools based entirely on their schema definitions, adding clear directives to your system prompt acts as an important guardrail. It prevents the model from attempting to use internal, sometimes inaccurate mental math and forces it to rely on your deterministic backend code instead.

Here is the updated script with a hardened system prompt designed to guide tool usage consistency:

```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Enhanced system prompt instructing Claude to delegate tasks to its tools
system_prompt = (
    "You are a precise mathematical assistant. You have access to specialized "
    "calculation tools. To ensure absolute accuracy and prevent arithmetic mistakes, "
    "you MUST always call the appropriate tool when performing any mathematical calculation, "
    "equation solving, or numerical reasoning task. Do not perform mental math."
)

# Load the tool schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a message requesting a calculation
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# Send the request with tools enabled
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas
)

# Print the complete response structure
print(json.dumps(response.model_dump(), indent=2))

```

### Prompt Hardening Best Practices:

* **The "MUST" Constraint:** Using definitive imperative language like *“you MUST always call the appropriate tool”* hardens Claude's decision matrix, significantly lowering the chance of the model dropping back into simple conversational text.
* **Providing Intent Nuance:** Explaining *why* it should use the tool (*“To ensure absolute accuracy and prevent arithmetic mistakes”*) helps Claude weigh the utility of its schemas when it encounters complex, multi-step word problems or ambiguous user inputs down the line!

## Parsing Claude's Tool Use Responses

## Observing Multiple Tool Requests Together

## Using Different Tool Types Together

## Using Different Tool Types Together